# Fixed-Size Chunking + Contextual Retrieval

**Part of:** RAG chunking-strategy evaluation for the "AI Engineering" book chatbot project — comparing
fixed-size, single-sentence, and 3-sentence chunking to see which produces the best retrieval quality.

**What this notebook does:**
1. Loads the "AI Engineering" PDF (Chip Huyen) page by page
2. Splits the text into fixed-size chunks (512 characters, 50-character overlap) using LangChain's `RecursiveCharacterTextSplitter`
3. Detects chapter boundaries from page footers, to group pages into sections
4. Adds "contextual retrieval" — for each chunk, a local LLM (via Ollama) generates a short 1-2 sentence description of where that chunk fits within its chapter, prepended to the chunk before embedding. This helps chunks that are meaningless in isolation (e.g. a bare table-of-contents fragment) become retrievable.
5. Runs contextual retrieval on a first-5-chapters subset (for fast evaluation against the other chunking strategies) as well as the full chunk set (for the production index)

**System-dependent settings — adjust for your own machine:**
- Developed on a laptop with an RTX 3050 (6GB VRAM). If you have more VRAM, you can safely run more concurrent Ollama requests, or use a larger model.
- The Ollama model used here (`llama3`, 8B) took roughly ~15s/chunk with the current few-shot prompt on this hardware — expect faster/slower depending on your GPU.

## Load the PDF

Loads the source PDF page-by-page using LangChain's `PyPDFLoader`. Each page becomes one `Document` object
with the page text plus metadata (page number, title, source path, etc.).

**Adjust:** update `file_path` below to point at your own PDF.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from pathlib import Path

D:\pytorch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
file_path = Path("../../data/raw/AI Engineering.pdf")
loader= PyPDFLoader(file_path)
docs= loader.load()

docs[25]

Document(metadata={'producer': 'Antenna House PDF Output Library 2.6.0 (Linux64)', 'creator': 'AH CSS Formatter V6.0 MR2 for Linux64 : 6.0.2.5372 (2012/05/16 18:26JST)', 'creationdate': '2024-12-04T13:39:11+00:00', 'author': 'Chip Huyen;', 'moddate': '2024-12-04T09:21:26-05:00', 'title': 'AI Engineering', 'trapped': '/False', 'ebx_publisher': "O'Reilly Media", 'source': '..\\..\\data\\raw\\AI Engineering.pdf', 'total_pages': 535, 'page': 25, 'page_label': '2'}, page_content='1 In this book, I use traditional ML to refer to all ML before foundation models.\nlarge-scale, readily available models brings about new possibilities and new chal‐\nlenges, which are the focus of this book.\nThis chapter begins with an overview of foundation models, the key catalyst behind\nthe explosion of AI engineering. I’ll then discuss a range of successful AI use cases,\neach illustrating what AI is good and not yet good at. As AI’s capabilities expand\ndaily, predicting its future possibilities becomes inc

In [4]:
docs[25].page_content[:200]

'1 In this book, I use traditional ML to refer to all ML before foundation models.\nlarge-scale, readily available models brings about new possibilities and new chal‐\nlenges, which are the focus of this'

In [5]:
len(docs)

535

## Split into fixed-size chunks

Uses LangChain's `RecursiveCharacterTextSplitter` to break each page into fixed-length chunks, splitting on
paragraph/sentence boundaries where possible before falling back to a hard character cut.

**Adjust:** `chunk_size` and `chunk_overlap` control retrieval granularity — smaller chunks are more precise
but lose surrounding context; larger chunks keep more context but can dilute relevance. 512/50 is a starting
point, not a rule — tune based on what retrieval quality looks like for your use case.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,        # characters per chunk — tune based on what retrieval size works for you
    chunk_overlap=50,      # overlap so context isn't lost at chunk boundaries
    
)

splits = splitter.create_documents(
    texts=[doc.page_content for doc in docs],
    metadatas=[doc.metadata for doc in docs],
)

In [7]:
splits[50]

Document(metadata={'producer': 'Antenna House PDF Output Library 2.6.0 (Linux64)', 'creator': 'AH CSS Formatter V6.0 MR2 for Linux64 : 6.0.2.5372 (2012/05/16 18:26JST)', 'creationdate': '2024-12-04T13:39:11+00:00', 'author': 'Chip Huyen;', 'moddate': '2024-12-04T09:21:26-05:00', 'title': 'AI Engineering', 'trapped': '/False', 'ebx_publisher': "O'Reilly Media", 'source': '..\\..\\data\\raw\\AI Engineering.pdf', 'total_pages': 535, 'page': 9, 'page_label': 'viii'}, page_content='Memory Bottlenecks                                                                                                     319\nBackpropagation and Trainable Parameters                                                          320\nMemory Math                                                                                                             322\nNumerical Representations                                                                                      325')

In [8]:
len(splits)

2562

## Contextual Retrieval

Individual fixed-size chunks are often meaningless out of context (e.g. a lone table-of-contents fragment,
or a chunk that starts mid-sentence). This section generates a short 1-2 sentence description situating each
chunk within its chapter, using a local LLM via Ollama, and prepends it to the chunk before embedding — based
on Anthropic's "contextual retrieval" technique.

Steps below: (1) detect chapter boundaries from page footers, (2) filter to the first 5 chapters for a faster
evaluation run against the other chunking strategies, (3) load the shared prompt template, (4) run contextual
retrieval with checkpointing — both on the 5-chapter subset and, separately, on the full chunk set.

### Detect chapter boundaries

Reconstructs chapter labels per page from the running footer text (e.g. `"2 | Chapter 1: ..."`), since the
PDF loader gives pages with no chapter metadata. Contextual retrieval uses this to give the LLM the whole
chapter as context, not just the isolated page.

**Caveat:** this regex is tuned to this PDF's specific footer format. Using a different source document?
Check the printed section list below — if it looks wrong (missing chapters, mismatched page counts), the
regex in `detect_chapter()` will need adjusting to match your document's footer/header pattern.

In [9]:
import re

def detect_chapter(page_text: str) -> str | None:
    """Extract chapter title from the running footer, if present."""
    match = re.search(r'\|\s*(Chapter\s+\d+[:.].*)$', page_text.strip())
    if match:
        return match.group(1).strip()
    return None

# Build page -> chapter label, with a fallback bucket for front matter (TOC, preface, etc.)
page_chapters = []
current_chapter = "Front Matter"
for doc in docs:
    detected = detect_chapter(doc.page_content)
    if detected:
        current_chapter = detected
    page_chapters.append(current_chapter)

# Group pages by chapter into section_text blocks
from collections import defaultdict
chapter_pages = defaultdict(list)
for doc, chapter in zip(docs, page_chapters):
    chapter_pages[chapter].append(doc.page_content)

section_text_by_chapter = {
    chapter: "\n".join(pages) for chapter, pages in chapter_pages.items()
}

# Map each page number -> its full section text (for context lookup)
page_to_section_text = {
    doc.metadata["page"]: section_text_by_chapter[chapter]
    for doc, chapter in zip(docs, page_chapters)
}

print(f"Detected {len(chapter_pages)} sections")
for ch, pages in list(chapter_pages.items()):
    print(f"  {ch!r} — {len(pages)} pages")

Detected 11 sections
  'Front Matter' — 25 pages
  'Chapter 1: Introduction to Building AI Applications with Foundation Models' — 48 pages
  'Chapter 2: Understanding Foundation Models' — 64 pages
  'Chapter 3: Evaluation Methodology' — 46 pages
  'Chapter 4: Evaluate AI Systems' — 52 pages
  'Chapter 5: Prompt Engineering' — 42 pages
  'Chapter 6: RAG and Agents' — 54 pages
  'Chapter 7: Finetuning' — 56 pages
  'Chapter 8: Dataset Engineering' — 42 pages
  'Chapter 9: Inference Optimization' — 44 pages
  'Chapter 10: AI Engineering Architecture and User Feedback' — 62 pages


### Filter to first 5 chapters (faster evaluation)

Contextual retrieval via a local LLM is slow. Rather than running it against the full ~2,500+ chunk set
before knowing if this chunking strategy is even competitive, this restricts a first pass to the first 5
chapters — matched against the same page range used for the single-sentence and 3-sentence chunking
notebooks, so the comparison is apples-to-apples.

**Note:** the full `splits` set (all chapters) is still used for the production run further below — this
`splits_first5` subset is only for the evaluation pass.

In [10]:
seen_chapters = []
for ch in page_chapters:
    if ch != "Front Matter" and ch not in seen_chapters:
        seen_chapters.append(ch)
    if len(seen_chapters) == 5:
        break

first_5_chapters = set(seen_chapters)
print(first_5_chapters)

docs_first5 = [
    doc for doc, ch in zip(docs, page_chapters)
    if ch in first_5_chapters
]
print(f"Filtered from {len(docs)} pages to {len(docs_first5)} pages")

splits_first5 = splitter.create_documents(
    texts=[doc.page_content for doc in docs_first5],
    metadatas=[doc.metadata for doc in docs_first5],
)
print(f"{len(splits_first5)} chunks in first 5 chapters (fixed-size)")

{'Chapter 3: Evaluation Methodology', 'Chapter 4: Evaluate AI Systems', 'Chapter 1: Introduction to Building AI Applications with Foundation Models', 'Chapter 5: Prompt Engineering', 'Chapter 2: Understanding Foundation Models'}
Filtered from 535 pages to 252 pages
1213 chunks in first 5 chapters (fixed-size)


### Load the shared context-generation prompt

The prompt template lives in a separate `.py` file (shared across all three chunking-strategy notebooks) so
it can be edited/iterated on without touching this notebook directly. Relies on being run from
`Ai_Engineering_Book_Chatbot/notebooks/.../` — if you move this notebook elsewhere, this relative path will
break.

In [11]:
import sys
from pathlib import Path

# Notebook cwd is assumed to be Ai_Engineering_Book_Chatbot/notebooks/Chunking/
prompts_dir = Path("../../Prompts/Contextual Retrieval/contextual Retrieval").resolve()
print(prompts_dir.exists())  # sanity check — should print True

sys.path.append(str(prompts_dir))
from prompt_v1 import CONTEXTUAL_RETRIEVAL_PROMPT

True


In [12]:
print(prompts_dir.exists())
from prompt_v1 import CONTEXTUAL_RETRIEVAL_PROMPT
print(CONTEXTUAL_RETRIEVAL_PROMPT[:200])  # sanity check it loaded correctly

True
You generate short retrieval contexts for chunks of text pulled from a larger document. Given a section of the document and a specific chunk from within it, write a 1-2 sentence context that situates 


### Ollama contextual-retrieval setup

**Adjust these for your own machine:**
- `OLLAMA_MODEL` — set to whatever model you've pulled locally (`ollama list` to check, `ollama pull <name>`
  to fetch one). Larger models give better context descriptions but run slower.
- `CHECKPOINT_PATH` — use a different filename depending on which run this is (e.g. the 5-chapter evaluation
  subset vs. the full-book production run), so they don't overwrite each other's progress.
- `CHECKPOINT_EVERY` — how many chunks between progress saves to disk. Lower = safer against crashes/kernel
  restarts, negligible extra cost given each checkpoint write is tiny.
- **Concurrency:** the loop below calls Ollama sequentially — one chunk at a time, no parallel requests. This
  is intentional: on a 6GB-VRAM GPU, running multiple 7B/8B-model requests concurrently risks running out of
  VRAM or just thrashing rather than speeding anything up. If your GPU has significantly more VRAM (24GB+),
  you could look into Ollama's `OLLAMA_NUM_PARALLEL` environment variable to parallelize this.
- `section_text[:6000]` truncation inside `generate_context()` — caps how much chapter text gets sent to the
  model per call. Raise this if your model has a larger context window and you want richer context; lower it
  to speed up inference on constrained hardware.

In [13]:
import ollama
import json
from pathlib import Path

from prompt_v1 import CONTEXTUAL_RETRIEVAL_PROMPT

OLLAMA_MODEL = "llama3"
CHECKPOINT_PATH = Path("contextual_splits_fixed_first5_checkpoint.jsonl")
CHECKPOINT_EVERY = 100

def generate_context(section_text: str, chunk_text: str) -> str:
    prompt = CONTEXTUAL_RETRIEVAL_PROMPT.format(
        section_text=section_text[:6000],
        chunk_text=chunk_text,
    )
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.0},
    )
    result = response["message"]["content"].strip()
    if result.lower().startswith("context:"):
        result = result[len("context:"):].strip()
    return result

def load_checkpoint():
    if not CHECKPOINT_PATH.exists():
        return {}
    done = {}
    with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            done[row["index"]] = row["context"]
    return done

def append_checkpoint(rows):
    with open(CHECKPOINT_PATH, "a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")

### Sanity-check on one chunk before running the full loop

Always test on a single chunk first — confirms the Ollama call, prompt formatting, and output parsing all
work, and gives a rough per-chunk timing estimate before committing to a multi-hour run.

In [14]:
import time

test_chunk = splits[50]
test_page = test_chunk.metadata.get("page")
test_section = page_to_section_text.get(test_page, "")

start = time.time()
result = generate_context(test_section, test_chunk.page_content)
print(result)
print(f"\nTook {time.time() - start:.1f}s")

Here is the context for the given chunk:

This chunk is from the book "AI Engineering: Building Applications with Foundation Models" by Chip Huyen, specifically the table of contents listing subsection titles and page numbers related to memory and training mechanics in a chapter on model training.

Took 64.8s


### Run contextual retrieval (long-running step)

This can take hours depending on chunk count, model size, and hardware — expect roughly 5 hours for the
~2,500-chunk full-book run on an RTX 3050 with an 8B model. Safe to interrupt anytime — progress is
checkpointed to disk every `CHECKPOINT_EVERY` chunks, and already-processed chunks are automatically skipped
if you restart the kernel and re-run this cell.

Run this once against `splits_first5` (for evaluation, with `CHECKPOINT_PATH` pointed at a `_first5`
checkpoint file), and separately against the full `splits` (for the production index).

In [15]:
from tqdm import tqdm

CHECKPOINT_PATH = Path("contextual_splits_fixed_first5_checkpoint.jsonl")

# Sanity check — catches this exact bug before it silently runs the wrong scope again
assert len(splits_first5) < len(splits), "splits_first5 should be a subset of splits"
print(f"About to process {len(splits_first5)} chunks (first 5 chapters)")

already_done = load_checkpoint()
print(f"Resuming — {len(already_done)} chunks already processed")

buffer = []
for i, chunk in enumerate(tqdm(splits_first5, desc="Generating context")):
    if i in already_done:
        continue

    page_num = chunk.metadata.get("page")
    section_text = page_to_section_text.get(page_num, "")

    try:
        context = generate_context(section_text, chunk.page_content)
    except Exception as e:
        print(f"Failed on chunk {i}: {e}")
        context = ""

    buffer.append({"index": i, "context": context})

    if len(buffer) >= CHECKPOINT_EVERY:
        append_checkpoint(buffer)
        buffer = []

if buffer:
    append_checkpoint(buffer)

print("Done.")

About to process 1213 chunks (first 5 chapters)
Resuming — 1212 chunks already processed


Generating context: 100%|█████████████████████████████████████████████████████████| 1213/1213 [00:02<00:00, 414.91it/s]

Done.


## Load contexts and build the contextual chunk list

In [16]:
import json
from pathlib import Path

FIXED_FIRST5_CHECKPOINT = Path("contextual_splits_fixed_first5_checkpoint.jsonl")

def load_contexts(checkpoint_path: Path) -> dict:
    contexts = {}
    with open(checkpoint_path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            contexts[row["index"]] = row["context"]
    return contexts

fixed_contexts = load_contexts(FIXED_FIRST5_CHECKPOINT)
print(f"Loaded {len(fixed_contexts)} contexts")

# Build final contextual text per chunk (context + original), keep chunk_id + metadata aligned
fixed_contextual_chunks = []
for i, chunk in enumerate(splits_first5):
    context = fixed_contexts.get(i, "")
    contextual_text = f"{context}\n\n{chunk.page_content}" if context else chunk.page_content
    fixed_contextual_chunks.append({
        "chunk_id": f"fixed_{i}",
        "page": chunk.metadata.get("page"),
        "page_label": chunk.metadata.get("page_label"),
        "original_text": chunk.page_content,
        "context": context,
        "contextual_text": contextual_text,
    })

print(f"{len(fixed_contextual_chunks)} contextual chunks ready for embedding")

Loaded 1213 contexts
1213 contextual chunks ready for embedding


## Set up the embedding model

In [17]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

## Set up Chroma and add the chunks

In [18]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db")

fixed_collection = chroma_client.get_or_create_collection(
    name="fixed_size_first5",
    metadata={"chunking_method": "fixed_size", "scope": "first_5_chapters"},
)

texts = [c["contextual_text"] for c in fixed_contextual_chunks]
ids = [c["chunk_id"] for c in fixed_contextual_chunks]
metadatas = [
    {"page": c["page"], "page_label": c["page_label"] or "", "original_text": c["original_text"]}
    for c in fixed_contextual_chunks
]

embeddings = embedding_model.encode(texts, show_progress_bar=True, batch_size=32).tolist()

fixed_collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=texts,
    metadatas=metadatas,
)

print(f"Added {fixed_collection.count()} chunks to Chroma collection 'fixed_size_first5'")

Batches: 100%|█████████████████████████████████████████████████████████████████████████| 38/38 [00:34<00:00,  1.11it/s]


Added 1213 chunks to Chroma collection 'fixed_size_first5'


## Quick sanity-check query

In [19]:
def query_fixed(question: str, top_k: int = 5):
    query_embedding = embedding_model.encode([question]).tolist()
    results = fixed_collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
    )
    return results

test_results = query_fixed("What does perplexity measure?")
for i, (doc, meta, dist) in enumerate(zip(
    test_results["documents"][0],
    test_results["metadatas"][0],
    test_results["distances"][0],
)):
    print(f"[{i+1}] (page {meta['page']}, dist={dist:.3f})")
    print(doc[:200])
    print()

[1] (page 145, dist=0.599)
This chunk is from Chapter 2 (Understanding Foundation Models), specifically the section on training data and its impact on foundation model performance.

What’s considered a good value for perplexity

[2] (page 221, dist=0.785)
This chunk is from Chapter 4 (Evaluate AI Systems) of the document, specifically the section on evaluation criteria and domain-specific capability. The chunk discusses the importance of evaluating AI 

[3] (page 145, dist=0.806)
This chunk is from Chapter 2: Understanding Foundation Models, specifically the section on Training Data, discussing the importance of data quality and the challenges of collecting sufficient training

[4] (page 144, dist=0.807)
This chunk is from Chapter 2 (Understanding Foundation Models), specifically the section on training data and its impact on foundation model performance.

bits) using 3.43 bits, this language model ca

[5] (page 146, dist=0.810)
This chunk is from Chapter 2 (Understanding Foundation Mo